# Tutorial 1: Building and Exploring PSP Knowledge Graphs

**ARIA: Causal-Aware Reasoning for Materials Discovery**

---

## Introduction

ARIA operates over **Processing-Structure-Property (PSP)** knowledge graphs that encode causal relationships in materials science. Unlike flat retrieval that simply fetches relevant documents, ARIA organizes knowledge into a **causal hierarchy** with three layers:

- **Processing (P):** Synthesis conditions (temperature, pressure, atmosphere, doping, method)
- **Structure (S):** Resulting material structure (crystallinity, defect density, grain size, phase)
- **Property (Pr):** Measurable properties (carrier mobility, conductivity, band gap)

The key insight is: **retrieval without causal completeness can harm reasoning.** A system that retrieves evidence about temperature and carrier mobility but has no structural explanation (the "Structure" layer) will produce unreliable predictions. ARIA gates evidence activation on whether the retrieved paths traverse all required PSP layers.

### What You Will Learn

1. Loading and inspecting a PSP knowledge graph
2. Understanding node types and edge classifications
3. Running KG diagnostics
4. Visualizing the graph structure
5. Identifying PSP-complete vs. incomplete paths

---

## 1. Setup

In [ ]:
import networkx as nx
import json

# ARIA imports
from aria import load_kg
from aria.kg.graph_store import kg_stats
from aria.kg.diagnostics import KGDiagnostics
from aria.kg.schema import (
    classify_node_layer,
    classify_path_layers,
    psp_layers_covered,
    PSPType,
)
from aria.retrieval.path_search import find_psp_paths, extract_mechanisms
from aria.retrieval.completeness import (
    causal_completeness_score,
    per_path_completeness,
    identify_missing_layers,
    PSPLayer,
)
from aria.visualization.graph_viz import plot_kg

## 2. Loading the Demo Knowledge Graph

ARIA loads knowledge graphs from enriched JSON files using `load_kg()`. The demo KG (`data/aria_2d_kg_tiny.json`) contains a small set of causal relationships for 2D materials (MoS2, WS2, etc.). If the file does not exist yet, we can build a tiny KG programmatically from the test fixtures.

In [ ]:
import os

# Try loading from file; fall back to building from the test fixture
kg_path = os.path.join("..", "data", "aria_2d_kg_tiny.json")

if os.path.exists(kg_path):
    kg = load_kg(kg_path)
    print(f"Loaded KG from file: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")
else:
    # Build a tiny KG from scratch (same structure as the test fixture)
    from aria.types import PSPRelationship, PSPType
    kg = nx.DiGraph()

    # Processing -> Structure edges
    kg.add_edge(
        "CVD temperature 750C", "improved crystallinity",
        mechanism="High CVD temperature promotes ordered MoS2 crystal growth",
        affected_property="crystallinity",
        confidence=0.9,
        psp_type=PSPType.PROCESSING_TO_STRUCTURE.value,
        relation="increases",
    )
    kg.add_edge(
        "CVD temperature 750C", "reduced defect density",
        mechanism="Elevated temperature anneals sulfur vacancies",
        affected_property="defect density",
        confidence=0.85,
        psp_type=PSPType.PROCESSING_TO_STRUCTURE.value,
        relation="decreases",
    )
    kg.add_edge(
        "doping concentration Nb", "doping_level",
        mechanism="Nb substitution at Mo sites introduces carriers",
        affected_property="doping_level",
        confidence=0.92,
        psp_type=PSPType.PROCESSING_TO_STRUCTURE.value,
        relation="increases",
    )
    kg.add_edge(
        "CVD temperature 750C", "grain growth",
        mechanism="Higher temperature increases grain size",
        affected_property="grain_size",
        confidence=0.8,
        psp_type=PSPType.PROCESSING_TO_STRUCTURE.value,
        relation="increases",
    )

    # Structure -> Structure edges
    kg.add_edge(
        "grain growth", "improved crystallinity",
        mechanism="Larger grains reduce grain boundary scattering",
        affected_property="crystallinity",
        confidence=0.75,
        psp_type=PSPType.STRUCTURE_TO_STRUCTURE.value,
        relation="increases",
    )

    # Structure -> Property edges
    kg.add_edge(
        "improved crystallinity", "higher carrier mobility",
        mechanism="Ordered crystal lattice enables efficient carrier transport",
        affected_property="carrier mobility",
        confidence=0.88,
        psp_type=PSPType.STRUCTURE_TO_PROPERTY.value,
        relation="increases",
    )
    kg.add_edge(
        "reduced defect density", "higher carrier mobility",
        mechanism="Fewer scattering centres boost mobility",
        affected_property="carrier mobility",
        confidence=0.82,
        psp_type=PSPType.STRUCTURE_TO_PROPERTY.value,
        relation="increases",
    )

    print(f"Built KG programmatically: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")

## 3. Exploring the KG Structure

Let us examine the nodes, edges, and their classifications in the PSP hierarchy.

In [ ]:
# Basic statistics
print("=" * 60)
print("Knowledge Graph Overview")
print("=" * 60)
print(f"  Nodes:  {kg.number_of_nodes()}")
print(f"  Edges:  {kg.number_of_edges()}")
print(f"  Density: {nx.density(kg):.4f}")
print(f"  Is DAG:  {nx.is_directed_acyclic_graph(kg)}")
print()

# Classify nodes by PSP layer
print("Node Classification by PSP Layer:")
print("-" * 40)
layer_counts = {"Processing": [], "Structure": [], "Property": [], "Unknown": []}
for node in kg.nodes():
    layer = classify_node_layer(node)
    layer_counts[layer or "Unknown"].append(node)

for layer, nodes in layer_counts.items():
    print(f"  {layer}: {len(nodes)} nodes")
    for n in nodes[:5]:
        print(f"    - {n}")
    if len(nodes) > 5:
        print(f"    ... and {len(nodes) - 5} more")

In [ ]:
# Examine edge types (PSP relationships)
print("Edge Types (PSP Relationships):")
print("-" * 40)

edge_type_counts = {}
for u, v, data in kg.edges(data=True):
    psp_type = data.get("psp_type", "unknown")
    edge_type_counts[psp_type] = edge_type_counts.get(psp_type, 0) + 1

for psp_type, count in sorted(edge_type_counts.items()):
    print(f"  {psp_type}: {count} edges")

print()
print("Edge Details:")
for u, v, data in kg.edges(data=True):
    print(f"  {u} -> {v}")
    print(f"    psp_type:    {data.get('psp_type', 'N/A')}")
    print(f"    relation:    {data.get('relation', 'N/A')}")
    print(f"    confidence:  {data.get('confidence', 'N/A')}")
    mech = data.get('mechanism', '')
    print(f"    mechanism:   {mech[:80]}{'...' if len(mech) > 80 else ''}")
    print()

### Understanding PSP Edge Types

ARIA classifies edges into five types:

| Edge Type | Description | Example |
|-----------|-------------|--------|
| `Processing_to_Structure` | Synthesis condition causes structural change | CVD temperature 750C -> improved crystallinity |
| `Structure_to_Property` | Structural feature determines property | improved crystallinity -> higher carrier mobility |
| `Processing_to_Property` | Direct shortcut (skips Structure) | doping Nb -> increased conductivity |
| `Structure_to_Structure` | Structural feature influences another | grain growth -> improved crystallinity |
| `Processing_to_Processing` | Process parameter influences another | annealing time -> temperature profile |

The most important edges are **Processing -> Structure -> Property**, which form causally complete PSP chains. The `Processing_to_Property` shortcut edges bypass the Structure layer and are considered **incomplete** by ARIA's causal completeness metric.

## 4. Running KG Diagnostics

The `KGDiagnostics` class provides comprehensive quality analysis of a knowledge graph, covering structure, content, query coverage, semantic diversity, and gap estimation.

In [ ]:
# Create a diagnostics instance and generate a full report
diag = KGDiagnostics(kg)
report = diag.generate_report()

# Print the formatted report
diag.print_report(report)

### Understanding the Diagnostic Report

The report covers five dimensions:

1. **Graph Structure:** Node/edge counts, density, DAG check, connected components. A well-structured PSP KG should be a DAG (no cycles) with clear root nodes (Processing) and leaf nodes (Property).

2. **Content Quality:** Mechanism coverage (what fraction of edges have textual evidence?), property annotation, average confidence.

3. **Query Coverage:** For representative materials-science queries, what fraction have at least one causal path in the KG?

4. **Semantic Diversity:** How diverse are the node embeddings? High diversity means the KG covers a broad range of concepts.

5. **Gap Estimation:** How many more edges/papers are needed to improve coverage?

In [ ]:
# Examine specific parts of the report in detail
structure = report["structure"]
print("Structural Highlights:")
print(f"  Root nodes (Processing inputs): {structure['root_nodes']}")
print(f"  Leaf nodes (Property outputs):   {structure['leaf_nodes']}")
print(f"  Intermediate (Structure):        {structure['intermediate_nodes']}")
print(f"  Longest causal path:             {structure['longest_path_length']} hops")

## 5. Visualizing the Knowledge Graph

ARIA provides a built-in visualization function that renders the KG with the JHU color theme. Nodes are sized by label length, and edges are annotated with mechanism text when available.

In [ ]:
# Visualize the KG (saves to file or displays interactively)
plot_kg(
    kg,
    output_path="kg_visualization.png",
    title="Demo PSP Causal Knowledge Graph",
    figsize=(16, 12),
    dpi=150,
)
print("KG visualization saved to kg_visualization.png")

## 6. Path Search and PSP Completeness

The core of ARIA's retrieval is finding causal paths in the KG. Let us search for paths and evaluate their **causal completeness** -- whether they traverse all required PSP layers.

In [ ]:
# Search for forward prediction paths: from CVD temperature to mobility
paths = find_psp_paths(
    kg,
    start_keywords=["temperature", "CVD"],
    end_keywords=["mobility"],
    max_hops=4,
)

print(f"Found {len(paths)} causal path(s):")
for i, path in enumerate(paths, 1):
    print(f"\n  Path {i}: {' -> '.join(path)}")
    
    # Classify each node by PSP layer
    layer_class = classify_path_layers(path, kg)
    for layer, nodes in layer_class.items():
        if nodes:
            print(f"    {layer}: {nodes}")
    
    # Check which PSP layers are covered
    covered = psp_layers_covered(path, kg)
    print(f"    Layers covered: {covered}")

In [ ]:
# Extract mechanisms from the paths
mechanisms = extract_mechanisms(kg, paths)
print("Mechanisms found along paths:")
for mech in mechanisms:
    print(f"  {mech['source']} -> {mech['target']}")
    print(f"    Mechanism: {mech['mechanism']}")
    print(f"    Confidence: {mech['confidence']}")
    print(f"    Property: {mech['affected_property']}")
    print()

### Causal Completeness Scoring

ARIA's causal completeness score measures whether retrieved evidence covers all PSP layers required by a query:

$$C(E, q) = \frac{|L(E) \cap L_{\text{req}}(q)|}{|L_{\text{req}}(q)|}$$

where $L(E)$ is the set of PSP layers covered by the evidence and $L_{\text{req}}(q)$ is the set of layers required by the query. A score of 1.0 means all required layers are covered; 0.0 means none are.

In [ ]:
# Compute causal completeness for different queries
queries = [
    "Predict carrier mobility for CVD MoS2 at 750C",
    "What is the band gap of WS2?",
    "How does temperature affect crystallinity?",
]

for query in queries:
    score = causal_completeness_score(kg, paths, query)
    missing = identify_missing_layers(kg, paths, query)
    required = PSPLayer  # reference
    print(f"Query: '{query}'")
    print(f"  Causal completeness: {score:.2f}")
    print(f"  Missing layers: {[l.value for l in missing]}")
    print()

In [ ]:
# Per-path completeness
per_path = per_path_completeness(kg, paths, "Predict carrier mobility for CVD MoS2 at 750C")
print("Per-path completeness:")
for path, score in per_path:
    path_str = " -> ".join(path)
    print(f"  Score {score:.2f}: {path_str}")

## 7. What Makes a PSP-Complete Path?

A **PSP-complete** path is one that traverses all three causal layers:

```
Processing -> Structure -> Property
```

For example:
- **PSP-complete:** `CVD temperature 750C -> improved crystallinity -> higher carrier mobility`
- **Incomplete (missing Structure):** `CVD temperature 750C -> higher carrier mobility` (direct shortcut)
- **Incomplete (missing Processing):** `improved crystallinity -> higher carrier mobility` (no root cause)

**Why does this matter?** ARIA's central finding is that *retrieval without causal completeness can harm reasoning*. A system that retrieves a Processing -> Property shortcut edge without the intervening Structure explanation is making an unsupported leap. The Structure layer provides the causal mechanism that justifies the prediction.

When ARIA encounters incomplete evidence, it can:
1. **Downgrade to Tier 2** (analogical transfer from similar materials)
2. **Fall back to Tier 3** (pure LLM, no KG evidence)
3. **Report the missing layers** in `ARIAResult.missing_evidence`

In [ ]:
# Demonstrate the difference between PSP-complete and incomplete paths
print("PSP Layer Analysis of Causal Paths:")
print("=" * 60)

for i, path in enumerate(paths, 1):
    covered = psp_layers_covered(path, kg)
    is_complete = covered == {"Processing", "Structure", "Property"}
    print(f"\nPath {i}: {' -> '.join(path)}")
    print(f"  Layers covered: {covered}")
    print(f"  PSP-complete:    {'YES' if is_complete else 'NO'}")
    if not is_complete:
        missing = {"Processing", "Structure", "Property"} - covered
        print(f"  Missing layers:  {missing}")
        print(f"  Risk: Prediction based on this path alone lacks causal justification")

## 8. Summary and Key Takeaways

In this tutorial, we:

1. **Loaded a PSP knowledge graph** and inspected its structure (nodes, edges, PSP layers)
2. **Ran comprehensive diagnostics** covering structure, content, coverage, diversity, and gaps
3. **Visualized the graph** to see the causal structure
4. **Searched for causal paths** and evaluated their PSP completeness
5. **Computed causal completeness scores** that quantify whether evidence covers all required PSP layers

### The Key Insight

> **Retrieval without causal completeness can harm reasoning.** Simply finding *some* relevant edges is not enough. ARIA gates evidence activation on whether the retrieved paths form causally complete PSP chains. This is what distinguishes ARIA from naive RAG: the *structure* of the retrieved evidence matters as much as its *relevance*.

In the next tutorial, we will use this KG to make forward predictions with ARIA's 3-tier reasoning cascade.